# Graded Activity: Let's Create a Bag of Words model for Sarcasm Detection
In this activity, you will create a Bag of Words model to detect sarcasm in text data. This is __not__ a good choice for a sarcasm detection model, but it is a good exercise to understand how Bag of Words works.

Fill me in here.

So let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [2]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material.

### Data
Let's load a public dataset of headlines that have been curated as either __sarcastic__ or __not sarcastic__. The dataset we'll use is [publically available on Kaggle](https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection) and is also discussed in the publications:
1. Misra, Rishabh and Prahal Arora. "Sarcasm Detection using News Headlines Dataset." AI Open (2023).
2. Misra, Rishabh and Jigyasa Grover. "Sculpting Data for ML: The first act of Machine Learning." ISBN 9798585463570 (2021).

We've packaged the sarcasm dataset in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). We'll load the dataset using [the `MySarcasmCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MySarcasmCorpus) which returns [a `MySarcasmRecordCorpusModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySarcasmRecordCorpusModel) with the fields:
* The `records::Dict{Int, MySarcasmRecordModel}` field holds the original records data as a dictionary, where the keys of the dictionary correspond to the headline index, and the values are [instances of the `MySarcasmRecordModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySarcasmRecordModel). Each record has the following fields:
    * `issarcastic`: has a value of `1` if the record is sarcastic; otherwise, `0.`
    * `headline`: the headline of the article, unstructured text
    * `article_link`: link to the original news article. Useful in collecting supplementary data

* The `tokens::Dict{String, Int64}` field holds the vocabulary computed over the __entire dataset__ as a dictionary, where the dictionary's keys are the tokens (words) and the values of the index of the word. We assemble the `tokens` dictionary in alphabetical order. 
* The `inverse::Dict{Int64, String}` field is the inverse of the `tokens` dictionary, where the keys are the token indexes and the values are the tokens (words).

Let's call the `MySarcasmCorpus(...)` method to load the sarcasm dataset and assign it to the `corpusmodel:MySarcasmRecordCorpusModel` variable. 

In [3]:
corpusmodel = MySarcasmCorpus(); # this loads the corpus model, which contains the vocabulary and tokenization information


### Compute Maximum Pad Length
Before we start, we need to compute the maximum pad length for the Bag of Words model. This will ensure that all input sequences are of the same length, which is generally a nice to have, and a requirement in some cases, for many machine learning models. 

Iterate through each headline using [a for-loop](https://docs.julialang.org/en/v1/manual/control-flow/#For-Loops-1), compute its size using [the `length(...)` method](https://docs.julialang.org/en/v1/base/collections/#Base.length), and then save this length.  If the test length is longer than we've seen before, this becomes the new maximum pad length.


In [5]:
max_pad_length = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    max_pad_length = 0; # initialize: we have 0 length
    
    # test the length of each headline
    for i ∈ 1:number_of_records
        test_record_length = tokenize(corpusmodel.records[i].headline, corpusmodel.tokens) |> length; # tokenize, and calc the number of tokens
        if (test_record_length > max_pad_length)
            max_pad_length = test_record_length; # we've found a new longest headline!
            println("Found a new longest headline: $(i) with length: $(max_pad_length)"); # show the record number and length
        end
    end
    max_pad_length
end;

Found a new longest headline: 1 with length: 10
Found a new longest headline: 2 with length: 15
Found a new longest headline: 11 with length: 16
Found a new longest headline: 14 with length: 18
Found a new longest headline: 37 with length: 20
Found a new longest headline: 97 with length: 21
Found a new longest headline: 106 with length: 22
Found a new longest headline: 189 with length: 23
Found a new longest headline: 584 with length: 24
Found a new longest headline: 1238 with length: 25
Found a new longest headline: 1450 with length: 26
Found a new longest headline: 2147 with length: 31
Found a new longest headline: 7303 with length: 153


## Task 1: Tokenize the Headlines
In this task, we'll tokenize the headlines in the sarcasm dataset. Tokenization is the process of splitting text into individual words or tokens. We'll use [the `tokenize(...)` method exported by the `VLDataScienceMachineLearningPackage.jl` package](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.tokenize) to perform this task. 

We'll store both the tokenized headlines and the headline labels as [a NamedTuple instances](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) in the `tokenized_headlines_vector::Vector{NamedTuple}` variable. Each Named Tuple will have the following fields:
- `headline::Vector{Int64}`: the tokenized headline, where each token is represented by its index in the vocabulary
- `issarcastic::Int64`: the label of the headline, where `1` indicates sarcasm and `0` indicates no sarcasm

In [ ]:
tokenized_headlines_vector = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    tokenized_headlines_vector = Vector{NamedTuple}(); # initialize the vector of NamedTuples

end

## Task 2: Create Unsarcastic Bag of Words
In this task, we will create two unsarcastic Bag of Words models, where we split the unsarcastic samples into a testing set, and a control set. The control set will be used to create a Bag of Words model that we can compare against the unsarcastic Bag of Words model, i.e., to compute a self-similarity score between unsarcastic samples.

## Task 3: Similarity between the Sarcastic and Unsarcastic Bag of Words
Fill me in here.